In [1]:
import os
import torch
import random
import numpy as np
from torch.utils.data import Dataset
from torchvision.io import read_image

class SAM_Precessed_MHB(Dataset):
    _views_per_burn = 4
    def __init__(self, root_dir, train_mode = False, return_full_masks=True, return_burn_masks=True):
        self.root_dir = root_dir
        self.train_mode = train_mode
        if train_mode:
            self._pairs_per_burn = 1
        else:
            self._pairs_per_burn = self._views_per_burn ** 2
        self.labels = torch.tensor(np.load(os.path.join(root_dir, "labels.npy")), dtype=torch.float32)
        self.return_full_masks = return_full_masks
        self.return_burn_masks = return_burn_masks
        self._len = self.labels.numel() * self._pairs_per_burn
        self._burn_levels = self.labels.shape[1]
        self._pairs_per_human = self._burn_levels * self._pairs_per_burn

    def __len__(self):
        return self._len

    def __getitem__(self, idx):
        human_idx = idx // self._pairs_per_human
        burn_idx = idx % self._pairs_per_human // self._pairs_per_burn
        if self.train_mode:
            front_view_idx = random.randint(0, self._views_per_burn - 1)
            back_view_idx = random.randint(0, self._views_per_burn - 1)
        else:
            front_view_idx = idx % self._pairs_per_human % self._pairs_per_burn // self._views_per_burn
            back_view_idx = idx % self._pairs_per_human % self._pairs_per_burn % self._views_per_burn
        res = {"label": self.labels[human_idx, burn_idx]}
        if self.return_full_masks:
            res["full_masks"] = (
                read_image(os.path.join(self.root_dir, "full_masks", "front", f"{human_idx}_{burn_idx}_{front_view_idx}.png")), 
                read_image(os.path.join(self.root_dir, "full_masks", "back", f"{human_idx}_{burn_idx}_{back_view_idx}.png"))
            )
        if self.return_burn_masks:
            res["burn_masks"] = (
                read_image(os.path.join(self.root_dir, "burn_masks", "front", f"{human_idx}_{burn_idx}_{front_view_idx}.png")), 
                read_image(os.path.join(self.root_dir, "burn_masks", "back", f"{human_idx}_{burn_idx}_{back_view_idx}.png"))
            )
        return res

In [2]:
from torch.utils.data import Subset

full_set = SAM_Precessed_MHB("sam_precessed_MHB")
test_indices = torch.tensor([list(range((i*625+500)*full_set._pairs_per_human,(i*625+625)*full_set._pairs_per_human)) for i in range(8)]).flatten()

test_set = Subset(full_set, test_indices)


In [3]:
class BaselineTest(Dataset):
    def __init__(self, ds):
        self.ds = ds

    def __len__(self):
        return len(self.ds)

    def __getitem__(self, idx):
        sample = self.ds[idx]
        full = sample["full_masks"][0].to(torch.bool).sum().item()+sample["full_masks"][1].to(torch.bool).sum().item()
        burn = sample["burn_masks"][0].to(torch.bool).sum().item()+sample["burn_masks"][1].to(torch.bool).sum().item()
        label = sample["label"].item()
        return burn/full, label, idx

In [4]:
from torch.utils.data import DataLoader

dataloader = DataLoader(BaselineTest(test_set), num_workers=32, in_order=False)

In [5]:
from tqdm import tqdm

pre_list = torch.empty([1000,8,16])
label_list = torch.empty([1000,8,16])
for b in tqdm(dataloader):
    idx = b[2].item()
    human_idx = idx // full_set._pairs_per_human
    burn_idx = idx % full_set._pairs_per_human // full_set._pairs_per_burn
    pair_idx = idx % full_set._pairs_per_human % full_set._pairs_per_burn
    pre_list[human_idx,burn_idx,pair_idx] = b[0].item()
    label_list[human_idx,burn_idx,pair_idx] = b[1].item()

torch.save(pre_list, "sam_p_MHB_baseline_pre.pt")

100%|██████████| 128000/128000 [03:49<00:00, 556.92it/s]


In [6]:
from sklearn.metrics import r2_score
print("R2 Score:", r2_score(label_list.flatten(), pre_list.flatten()))

R2 Score: 0.9672377109527588


In [7]:
triclass_label = torch.zeros_like(label_list)
triclass_label[label_list < 0.05] = 0
triclass_label[(label_list >= 0.05) & (label_list < 0.2)] = 1
triclass_label[label_list >= 0.2] = 2
triclass_pre = torch.zeros_like(pre_list)
triclass_pre[pre_list < 0.05] = 0
triclass_pre[(pre_list >= 0.05) & (pre_list < 0.2)] = 1
triclass_pre[pre_list >= 0.2] = 2
from sklearn.metrics import classification_report
print(classification_report(triclass_label.flatten(), triclass_pre.flatten(), digits=3))

              precision    recall  f1-score   support

         0.0      0.961     0.867     0.912     15600
         1.0      0.915     0.910     0.912     32640
         2.0      0.971     0.991     0.981     79760

    accuracy                          0.955    128000
   macro avg      0.949     0.923     0.935    128000
weighted avg      0.955     0.955     0.955    128000



In [8]:
mae = (pre_list-label_list).abs()
print(f"MAE:{mae.mean().item()*100:.2f}, STD:{torch.std(mae).item()*100:.2f}")

MAE:2.36, STD:3.88


In [9]:
mre = (pre_list-label_list).abs()/(label_list)
print(f"MRE:{mre.mean().item()*100:.2f}, STD:{torch.std(mre).item()*100:.2f}")

MRE:10.94, STD:12.47


In [10]:
import pandas as pd
import pingouin as pg

reshaped_pre = pre_list.view(-1, full_set._pairs_per_burn)
n_subjects, n_raters = reshaped_pre.shape

long_data_list = []
for i in range(n_raters):
    rater_name = f'Observer_{i+1}'
    temp_df = pd.DataFrame({
        'subject': torch.arange(n_subjects),
        'rater': rater_name,
        'rating': reshaped_pre[:, i]
    })
    long_data_list.append(temp_df)

long_data = pd.concat(long_data_list, ignore_index=True)
pg.intraclass_corr(
    data=long_data,
    targets='subject',
    raters='rater',
    ratings='rating'
)

,Type,Description,ICC,F,df1,df2,pval,CI95
0,ICC1,Single raters absolute,0.983782,971.551129,7999,120000,0.0,"[0.98, 0.98]"
1,ICC2,Single random raters,0.983782,971.830571,7999,119985,0.0,"[0.98, 0.98]"
2,ICC3,Single fixed raters,0.983786,971.830571,7999,119985,0.0,"[0.98, 0.98]"
3,ICC1k,Average raters absolute,0.998971,971.551129,7999,120000,0.0,"[1.0, 1.0]"
4,ICC2k,Average random raters,0.998971,971.830571,7999,119985,0.0,"[1.0, 1.0]"
5,ICC3k,Average fixed raters,0.998971,971.830571,7999,119985,0.0,"[1.0, 1.0]"
